# MNIST 데이터셋으로 신경망 추론

이 노트북에서는 사전 학습된 가중치를 사용하여 MNIST 손글씨 숫자 이미지 인식 신경망을 구동합니다.

## 신경망 구조

| 층 | 뉴런 수 | 활성화 함수 |
|----|---------|------------|
| 입력층 | 784 | - |
| 은닉층 1 | 50 | sigmoid |
| 은닉층 2 | 100 | sigmoid |
| 출력층 | 10 | softmax |

---

## 📌 실습 과제: MNIST 신경망 추론

이 섹션의 코드는 이후 실습 과제에서 `import` 형태로 재사용됩니다.

In [ ]:
# coding: utf-8
# ============================================
# 실습 과제: MNIST 신경망 추론
# ============================================

# 필요한 라이브러리 Import
import os, sys, time
print(os.getcwd())

# 추가 Library import를 위한 Current dir 변경
# 현재 디렉토리(ch02)의 상위 디렉토리로 이동하여
# dataset와 common 모듈을 찾을 수 있도록 합니다
current_dir = os.path.dirname(os.getcwd())
print(current_dir)
os.chdir(current_dir)

# 추가 Library import
import numpy as np
import pickle
from dataset.mnist import load_mnist
from common.functions import sigmoid, softmax

In [ ]:
# ============================================
# 실습 과제: 데이터 로드 함수
# ============================================

def get_data():
    """
    MNIST 데이터를 로드하고 반환합니다.
    
    Returns
    -------
    tuple: (테스트 이미지, 테스트 레이블)
    """
    # 숫자 데이터 다운로드 from internet
    # normalize=True: 픽셀 값을 0.0~1.0으로 정규화
    # flatten=True: 이미지를 1차원 배열로 펼침 (784차원)
    # one_hot_label=False: 레이블을 정수 인덱스로 반환
    (x_train, t_train), (x_test, t_test) = load_mnist(
        normalize=True, flatten=True, one_hot_label=False
    )
    return x_test, t_test

In [ ]:
# ============================================
# 실습 과제: 신경망 초기화 함수
# ============================================

def init_network():
    """
    사전 학습된 가중치 파일을 로드하여 신경망을 초기화합니다.
    
    Returns
    -------
    dict: 가중치(W)와 편향(b)을 포함하는 신경망 파라미터
    """
    print(os.getcwd())
    
    # 신경망 가중치값 파일 로딩
    # sample_weight.pkl에는 학습된 가중치 W1, W2, W3과
    # 편향 b1, b2, b3이 pickle 형태로 저장되어 있습니다
    with open("ch02/sample_weight.pkl", "rb") as f:
        network = pickle.load(f)
    
    return network

In [ ]:
# ============================================
# 실습 과제: 예측 (순전파) 함수
# ============================================

def predict(network, x):
    """
    신경망을 통해 입력 데이터 x의 출력을 예측합니다.
    
    Parameters
    ----------
    network : dict
        초기화된 신경망 (가중치와 편향 포함)
    x : numpy array
        입력 데이터 (784차원 벡터)
    
    Returns
    -------
    numpy array: 출력층의 확률 분포 (10차원)
    """
    # 가중치와 편향 추출
    W1, W2, W3 = network["W1"], network["W2"], network["W3"]
    b1, b2, b3 = network["b1"], network["b2"], network["b3"]

    # 신경망 출력값 계산 (순전파)
    # 입력층 → 은닉층 1
    a1 = np.dot(x, W1) + b1
    z1 = sigmoid(a1)
    
    # 은닉층 1 → 은닉층 2
    a2 = np.dot(z1, W2) + b2
    z2 = sigmoid(a2)
    
    # 은닉층 2 → 출력층
    a3 = np.dot(z2, W3) + b3
    y = softmax(a3)

    return y

In [ ]:
# ============================================
# 실습 과제: 정확도 계산
# ============================================

# 데이터 로드
x, t = get_data()

# 신경망 초기화 (가중치 로드)
network = init_network()

# 정확도 카운터 초기화
accuracy_cnt = 0

# 테스트 데이터 전체에 대해 예측 수행
for i in range(len(x)):
    # 신경망 출력값 계산
    y = predict(network, x[i])
    
    # 확률이 가장 높은 원소의 인덱스를 얻는다 (예측 클래스)
    p = np.argmax(y)
    
    # 예측값과 정답이 맞는지 비교
    if p == t[i]:
        print(f"Predicted num= {p},  Original num= {t[i]}")
        time.sleep(0.5)
        # 정답을 맞춘 경우 1점 추가
        accuracy_cnt += 1

# 최종 정확도 출력
print(f"Accuracy: {str(float(accuracy_cnt) / len(x))}")

---

## 📖 설명: 신경망 추론 과정 이해하기

이 섹션에서는 MNIST 신경망이 어떻게 동작하는지 자세히 설명합니다.

### 전체 흐름 요약

```
1. 데이터 로드 (get_data)
   ↓
2. 신경망 초기화 (init_network) - 가중치 파일 로드
   ↓
3. 예측 수행 (predict) - 순전파
   ↓
4. 정확도 계산 - 예측값과 정답 비교
```

In [ ]:
# 데이터 로드 및 신경망 초기화
x, t = get_data()
network = init_network()

# 로드된 데이터와 신경망 정보 확인
print(f"테스트 이미지 개수: {len(x)}")
print(f"테스트 이미지 형태: {x[0].shape}")
print(f"테스트 레이블 형태: {t.shape}")

print("\n신경망 파라미터 정보:")
for key, value in network.items():
    print(f"  {key}: 형태 = {value.shape}")

In [ ]:
# 단일 이미지 예측 예시
img_idx = 0
sample_img = x[img_idx]
sample_label = t[img_idx]

# 예측 수행
y = predict(network, sample_img)

# 결과 확인
print(f"입력 이미지 인덱스: {img_idx}")
print(f"정답 레이블: {sample_label}")
print(f"예측 확률: {y}")
print(f"예측 클래스: {np.argmax(y)}")
print(f"최대 확률: {np.max(y):.4f}")

In [ ]:
# 여러 이미지 예측 결과 확인
import matplotlib.pyplot as plt

def show_predictions(network, x, t, num_show=10):
    """
    여러 이미지의 예측 결과를 함께 표시합니다.
    
    Parameters
    ----------
    network : dict
        초기화된 신경망
    x : numpy array
        테스트 이미지
    t : numpy array
        테스트 레이블
    num_show : 표시할 이미지 개수
    """
    fig, axes = plt.subplots(2, 5, figsize=(15, 5))
    correct = 0
    
    for i, ax in enumerate(axes.flat):
        if i >= num_show:
            break
            
        # 이미지 표시
        img = x[i].reshape(28, 28)
        ax.imshow(img, cmap='gray')
        
        # 예측 수행
        y = predict(network, x[i])
        predicted = np.argmax(y)
        actual = t[i]
        confidence = y[predicted]
        
        # 정답 여부 확인
        is_correct = predicted == actual
        if is_correct:
            correct += 1
        
        # 제목 설정 (예측: 실제, 확률, 정답/오답)
        status = "✓" if is_correct else "✗"
        ax.set_title(f"예측: {predicted} (실제: {actual})\n확률: {confidence:.2%} {status}")
        ax.axis('off')
    
    plt.suptitle(f"예측 결과 ({correct}/{num_show} 정답)", fontsize=14)
    plt.tight_layout()
    plt.show()

# 첫 10개 이미지 예측 결과 표시
show_predictions(network, x, t, 10)

### 순전파 (Forward Propagation) 상세

신경망의 각 층에서 발생하는 연산을 단계별로 확인합니다.

**수식:**
- 은닉층 1: `a1 = X @ W1 + b1`, `z1 = sigmoid(a1)`
- 은닉층 2: `a2 = z1 @ W2 + b2`, `z2 = sigmoid(a2)`
- 출력층: `a3 = z2 @ W3 + b3`, `y = softmax(a3)`

In [ ]:
# 순전파 과정 단계별 확인
sample_idx = 5
sample_x = x[sample_idx]
sample_t = t[sample_idx]

W1, W2, W3 = network["W1"], network["W2"], network["W3"]
b1, b2, b3 = network["b1"], network["b2"], network["b3"]

print(f"입력 이미지 인덱스: {sample_idx}")
print(f"정답 레이블: {sample_t}")
print(f"\n입력 형태: {sample_x.shape}")

# 은닉층 1
a1 = np.dot(sample_x, W1) + b1
z1 = sigmoid(a1)
print(f"\n은닉층 1:")
print(f"  가중합 a1 형태: {a1.shape}")
print(f"  활성화 z1 형태: {z1.shape}")
print(f"  z1 최소/최대: {z1.min():.4f} / {z1.max():.4f}")

# 은닉층 2
a2 = np.dot(z1, W2) + b2
z2 = sigmoid(a2)
print(f"\n은닉층 2:")
print(f"  가중합 a2 형태: {a2.shape}")
print(f"  활성화 z2 형태: {z2.shape}")
print(f"  z2 최소/최대: {z2.min():.4f} / {z2.max():.4f}")

# 출력층
a3 = np.dot(z2, W3) + b3
y = softmax(a3)
print(f"\n출력층:")
print(f"  가중합 a3: {a3}")
print(f"  출력 y (확률): {y}")
print(f"  예측 클래스: {np.argmax(y)}")
print(f"  확률 합계: {y.sum():.6f}")

### 전체 정확도 계산

테스트 데이터 10,000개에 대해 신경망이 몇 개를 맞췄는지 확인합니다.

In [ ]:
# 전체 테스트 데이터로 정확도 계산
x_all, t_all = get_data()
network_all = init_network()

correct_cnt = 0
total = len(x_all)

print(f"총 {total}개 이미지 예측 중...")

for i in range(total):
    y = predict(network_all, x_all[i])
    p = np.argmax(y)
    if p == t_all[i]:
        correct_cnt += 1
    
    # 진행 상황 표시 (1000개마다)
    if (i + 1) % 1000 == 0:
        print(f"  {i + 1}/{total} 처리 완료, 현재 정확도: {correct_cnt / (i + 1):.4f}")

final_accuracy = float(correct_cnt) / total
print(f"\n최종 정확도: {final_accuracy:.4f} ({correct_cnt}/{total})")
print(f"퍼센트: {final_accuracy * 100:.2f}%")

---

## 보충 자료: MNIST 신경망 이해하기

### 가중치 파일 (sample_weight.pkl)

pickle 파일에는 다음과 같은 딕셔너리 구조가 저장되어 있습니다:

```python
network = {
    'W1': (784, 50),   # 입력층 → 은닉층 1 가중치
    'b1': (50,),       # 은닉층 1 편향
    'W2': (50, 100),   # 은닉층 1 → 은닉층 2 가중치
    'b2': (100,),      # 은닉층 2 편향
    'W3': (100, 10),   # 은닉층 2 → 출력층 가중치
    'b3': (10,)        # 출력층 편향
}
```

### 클래스별 정확도 분석

각 숫자(0~9) 클래스별로 얼마나 잘 인식하는지 확인해볼 수 있습니다.

In [ ]:
# 클래스별 정확도 분석
x_cls, t_cls = get_data()
network_cls = init_network()

class_correct = [0] * 10
class_total = [0] * 10

for i in range(len(x_cls)):
    y = predict(network_cls, x_cls[i])
    p = np.argmax(y)
    actual = t_cls[i]
    class_total[actual] += 1
    if p == actual:
        class_correct[actual] += 1

print("클래스별 정확도:")
print(f"{'클래스':<8} {'총 개수':<10} {'정답 수':<10} {'정확도':<10}")
print("-" * 40)
for cls in range(10):
    acc = class_correct[cls] / class_total[cls] if class_total[cls] > 0 else 0
    print(f"{cls:<8} {class_total[cls]:<10} {class_correct[cls]:<10} {acc:<10.4f}")

print(f"\n전체 정확도: {sum(class_correct) / sum(class_total):.4f}")

### 주요 개념 요약

| 개념 | 설명 |
|------|------|
| **순전파 (Forward Propagation)** | 입력 → 은닉층 → 출력층 방향으로 신호가 전달되는 과정 |
| **가중치 (W)** | 뉴런 간 연결의 강도, 학습을 통해 조정 |
| **편향 (b)** | 뉴런의 활성화 임계값 |
| **sigmoid** | 은닉층 활성화 함수, 입력을 0~1 사이로 압축 |
| **softmax** | 출력층 활성화 함수, 확률 분포로 변환 |
| **argmax** | 가장 큰 값을 가진 인덱스 반환 (예측 클래스 결정) |
| **pickle** | Python 객체를 직렬화하여 파일에 저장/로드하는 형식 |